In [ ]:
import pandas as pd 
from tqdm import tqdm
import requests
import time
import random

### Perfil de deputados

In [ ]:
def make_api_request(max_retries=3, base_delay=1, max_delay=60, backoff_factor=2):
    """
    Make API request for deputados with retry logic and exponential backoff.
    """
    all_data = []
    page = 1
    items_per_page = 100

    with tqdm(desc="Fetching deputados data", unit="page") as pbar:
        while True:
            retry_count = 0
            success = False
            page_data = None

            while retry_count <= max_retries and not success:
                try:
                    response = requests.get(
                        url="https://dadosabertos.camara.leg.br/api/v2/deputados",
                        headers={
                            "Accept": "application/json",
                            "Content-Type": "application/json"
                        },
                        params={
                            "id": "",
                            'nome': '',
                            'idLegislatura': '',
                            "siglaUf": "",
                            "siglaPartido": "",
                            'siglaSexo': '',
                            'pagina': page,
                            'itens': items_per_page,
                            "dataInicio": "2003-02-01",
                            "dataFim": "2023-01-31",
                            "ordem": "ASC",
                            "ordenarPor": "nome"
                        },
                        timeout=30
                    )
                    response.raise_for_status()
                    data = response.json()
                    page_data = data.get('dados', [])

                    if not page_data:  # No data on this page
                        success = True
                        break

                    all_data.extend(page_data)
                    pbar.set_postfix({"Page": page, "Items": len(page_data), "Retries": retry_count})
                    pbar.update(1)

                    # If fewer items than requested, last page reached
                    if len(page_data) < items_per_page:
                        success = True
                        page_data = None  # Break outer loop
                        break

                    page += 1
                    success = True
                    time.sleep(0.5)

                except (requests.exceptions.HTTPError,
                        requests.exceptions.ConnectionError,
                        requests.exceptions.Timeout,
                        requests.exceptions.RequestException) as err:
                    retry_count += 1
                    if retry_count > max_retries:
                        print(f"\nFailed after {max_retries} retries on page {page}. Error: {err}")
                        if all_data:
                            print(f"Returning partial data: {len(all_data)} items from {page-1} pages")
                            return pd.json_normalize(all_data)
                        return None
                    delay = min(base_delay * (backoff_factor ** (retry_count - 1)), max_delay)
                    jitter = random.uniform(0, 0.1 * delay)
                    total_delay = delay + jitter
                    print(f"\nRetry {retry_count}/{max_retries} for page {page} after {total_delay:.1f}s. Error: {type(err).__name__}")
                    time.sleep(total_delay)

                except Exception as e:
                    print(f"\nUnexpected error on page {page}: {e}")
                    if all_data:
                        print(f"Returning partial data: {len(all_data)} items from {page-1} pages")
                        return pd.json_normalize(all_data)
                    return None

            # Break outer loop if last page reached
            if page_data is None:
                break

    if all_data:
        df = pd.json_normalize(all_data)
        print(f"\nRequest successful! Total items: {len(all_data)}")
        return df
    print("\nNo data fetched.")
    return None

# Usage
df_deputados = make_api_request(
    max_retries=5,
    base_delay=2,
    max_delay=120,
    backoff_factor=2
)

if df_deputados is not None:
    df_deputados.to_csv('../data/raw/deputados.csv', index=False)
    unique_ids = list(df_deputados['id'].unique())


### Dados de partidos

In [ ]:
def make_api_request(max_retries=3, base_delay=1, max_delay=60, backoff_factor=2):
    """
    Make API request for partidos with retry logic and exponential backoff.
    """
    all_data = []  # List to collect data from all pages
    page = 1
    items_per_page = 100  # Max items per page allowed by API

    # Create an indeterminate progress bar
    with tqdm(desc="Fetching partidos data", unit="page") as pbar:
        while True:
            retry_count = 0
            success = False
            page_data = None

            while retry_count <= max_retries and not success:
                try:
                    response = requests.get(
                        url="https://dadosabertos.camara.leg.br/api/v2/partidos",
                        headers={
                            "Accept": "application/json",
                            "Content-Type": "application/json"
                        },
                        params={
                            "sigla": "",
                            'idLegislatura': '',
                            'pagina': page,
                            'itens': items_per_page,
                            "dataInicio": "2003-02-01",
                            "dataFim": "2023-01-31",
                            "ordem": "ASC",
                            "ordenarPor": "sigla"
                        },
                        timeout=30
                    )
                    response.raise_for_status()
                    data = response.json()
                    page_data = data['dados']

                    if not page_data:  # No data at all
                        success = True
                        break

                    all_data.extend(page_data)
                    pbar.set_postfix({"Page": page, "Items": len(page_data), "Retries": retry_count})
                    pbar.update(1)

                    # If fewer items than requested, last page reached
                    if len(page_data) < items_per_page:
                        success = True
                        page_data = None  # Mark to break outer loop
                        break

                    page += 1
                    success = True
                    time.sleep(0.5)

                except (requests.exceptions.HTTPError, 
                        requests.exceptions.ConnectionError,
                        requests.exceptions.Timeout,
                        requests.exceptions.RequestException) as err:
                    retry_count += 1
                    if retry_count > max_retries:
                        print(f"\nFailed after {max_retries} retries on page {page}. Error: {err}")
                        break
                    delay = min(base_delay * (backoff_factor ** (retry_count - 1)), max_delay)
                    jitter = random.uniform(0, 0.1 * delay)
                    total_delay = delay + jitter
                    print(f"\nRetry {retry_count}/{max_retries} for page {page} after {total_delay:.1f}s. Error: {type(err).__name__}")
                    time.sleep(total_delay)

            if page_data is None:
                break

    if all_data:
        df = pd.json_normalize(all_data)
        print(f"\nRequest successful! Total items: {len(all_data)}")
        return df
    else:
        print("\nNo data fetched.")
        return None

# Usage with custom retry parameters
df_partidos = make_api_request(
    max_retries=5,      # Try up to 5 times per request
    base_delay=2,       # Start with 2 second delay
    max_delay=120,      # Cap delays at 2 minutes
    backoff_factor=2    # Double the delay each retry
)

if df_partidos is not None:
    df_partidos.to_csv('../data/raw/partidos.csv', index=False)  # Fixed filename

### Mudanças de Partidos

In [ ]:
def make_api_request(unique_ids, max_retries=3, base_delay=1, max_delay=60, backoff_factor=2):
    """
    Make API requests for deputados histórico with retry logic and exponential backoff.
    """
    all_data = []
    failed_ids = []

    for id in tqdm(unique_ids, desc="Fetching histórico data", unit="ID"):
        retry_count = 0
        success = False

        while retry_count <= max_retries and not success:
            try:
                response = requests.get(
                    url=f"https://dadosabertos.camara.leg.br/api/v2/deputados/{id}/historico",
                    headers={
                        "Accept": "application/json",
                        "Content-Type": "application/json"
                    },
                    params={
                        "dataInicio": "2003-02-01",
                        "dataFim": "2023-01-31",
                    },
                    timeout=30
                )
                response.raise_for_status()
                data = response.json()
                page_data = data.get('dados', [])

                if not page_data:
                    # No data found for this ID, treat as successful fetch
                    success = True
                    break

                # Add the ID to each record
                for record in page_data:
                    record['deputado_id'] = id

                all_data.extend(page_data)
                success = True
                # time.sleep(0.5)

            except (requests.exceptions.HTTPError,
                    requests.exceptions.ConnectionError,
                    requests.exceptions.Timeout,
                    requests.exceptions.RequestException) as err:
                retry_count += 1
                if retry_count > max_retries:
                    print(f"\nFailed after {max_retries} retries for ID {id}. Error: {err}")
                    failed_ids.append(id)
                    break
                delay = min(base_delay * (backoff_factor ** (retry_count - 1)), max_delay)
                jitter = random.uniform(0, 0.1 * delay)
                total_delay = delay + jitter
                print(f"\nRetry {retry_count}/{max_retries} for ID {id} after {total_delay:.1f}s. Error: {type(err).__name__}")
                time.sleep(total_delay)

            except Exception as e:
                retry_count += 1
                if retry_count > max_retries:
                    print(f"\nUnexpected error after {max_retries} retries for ID {id}: {e}")
                    failed_ids.append(id)
                    break
                delay = min(base_delay * (backoff_factor ** (retry_count - 1)), max_delay)
                jitter = random.uniform(0, 0.1 * delay)
                total_delay = delay + jitter
                print(f"\nUnexpected error retry {retry_count}/{max_retries} for ID {id} after {total_delay:.1f}s")
                time.sleep(total_delay)

    # Summary
    if all_data:
        df = pd.json_normalize(all_data)
        print(f"Total records collected: {len(all_data)}")
        return df, failed_ids
    else:
        print("No data fetched.")
        return None, failed_ids


# Fetch histórico data for all IDs
df_migracoes, failed_ids = make_api_request(
    unique_ids,       # Your list of deputado IDs
    max_retries=3,    # Retry up to 3 times initially
    base_delay=1,     # Start with 1 second delay
    max_delay=60,     # Max 60 seconds between retries
    backoff_factor=2  # Double delay on each retry
)

# Save results
if df_migracoes is not None:
    df_migracoes.to_csv('../data/raw/deputados_historico.csv', index=False)

### Discursos

In [ ]:
unique_ids

In [24]:
import requests
import time
import random
import pandas as pd
from tqdm import tqdm

def fetch_deputados_discursos(unique_ids, data_inicio=None, data_fim=None,
                              id_legislatura=None, ordenar_por="dataHoraInicio",
                              ordem="DESC", itens=100,
                              max_retries=3, base_delay=1, max_delay=60, backoff_factor=2):
    """
    Fetch discursos (speeches) of deputados from the Câmara API with retry logic,
    looping over legislatures and pages individually.
    Shows deputy progress in bar with detailed info in description.
    """
    all_data = []
    failed_ids = []

    if not id_legislatura:
        legislaturas = [None]
    elif isinstance(id_legislatura, (list, tuple)):
        legislaturas = id_legislatura
    else:
        legislaturas = [id_legislatura]

    # Main progress bar tracks deputies
    with tqdm(total=len(unique_ids), desc="Processing Deputies", unit="deputy") as pbar:
        
        for dep_idx, dep_id in enumerate(unique_ids):
            
            for leg_idx, leg in enumerate(legislaturas):
                pagina = 1
                
                while True:
                    # Update description with current status
                    leg_display = f"Leg {leg}" if leg else "All Legs"
                    desc = f"Deputy {dep_id} | {leg_display} (Leg {leg_idx+1}/{len(legislaturas)}) | Page {pagina} | Total Records: {len(all_data)}"
                    pbar.set_description(desc)
                    
                    retry_count = 0
                    success = False

                    while retry_count <= max_retries and not success:
                        try:
                            params = {
                                "ordenarPor": ordenar_por,
                                "ordem": ordem,
                                "itens": itens,
                                "pagina": pagina
                            }

                            if data_inicio:
                                params["dataInicio"] = data_inicio
                            if data_fim:
                                params["dataFim"] = data_fim
                            if leg:
                                params["idLegislatura"] = str(leg)

                            response = requests.get(
                                url=f"https://dadosabertos.camara.leg.br/api/v2/deputados/{dep_id}/discursos",
                                headers={
                                    "Accept": "application/json",
                                    "Content-Type": "application/json"
                                },
                                params=params,
                                timeout=30
                            )
                            response.raise_for_status()
                            data = response.json()
                            page_data = data.get("dados", [])

                            # Update description with found records
                            records_found = len(page_data)
                            desc_with_results = f"Deputy {dep_id} | {leg_display} (Leg {leg_idx+1}/{len(legislaturas)}) | Page {pagina} | Found: {records_found} | Total: {len(all_data)}"
                            pbar.set_description(desc_with_results)

                            # If empty, stop iterating this legislature
                            if not page_data:
                                success = True
                                break

                            # Add deputado ID and legislatura to each record
                            for record in page_data:
                                record["deputado_id"] = dep_id
                                record["idLegislatura"] = leg

                            all_data.extend(page_data)
                            success = True
                            pagina += 1  # next page

                        except (requests.exceptions.HTTPError,
                                requests.exceptions.ConnectionError,
                                requests.exceptions.Timeout,
                                requests.exceptions.RequestException) as err:
                            retry_count += 1
                            if retry_count > max_retries:
                                pbar.write(f"❌ Failed after {max_retries} retries for ID {dep_id}, Leg {leg}, Page {pagina}. Error: {err}")
                                failed_ids.append((dep_id, leg, pagina))
                                break
                            delay = min(base_delay * (backoff_factor ** (retry_count - 1)), max_delay)
                            jitter = random.uniform(0, 0.1 * delay)
                            total_delay = delay + jitter
                            
                            # Show retry in description
                            retry_desc = f"Deputy {dep_id} | {leg_display} | Page {pagina} | Retrying {retry_count}/{max_retries} in {total_delay:.1f}s"
                            pbar.set_description(retry_desc)
                            pbar.write(f"🔄 Retry {retry_count}/{max_retries} for ID {dep_id}, Leg {leg}, Page {pagina} after {total_delay:.1f}s. Error: {type(err).__name__}")
                            time.sleep(total_delay)

                        except Exception as e:
                            retry_count += 1
                            if retry_count > max_retries:
                                pbar.write(f"❌ Unexpected error after {max_retries} retries for ID {dep_id}, Leg {leg}, Page {pagina}: {e}")
                                failed_ids.append((dep_id, leg, pagina))
                                break
                            delay = min(base_delay * (backoff_factor ** (retry_count - 1)), max_delay)
                            jitter = random.uniform(0, 0.1 * delay)
                            total_delay = delay + jitter
                            
                            # Show retry in description
                            retry_desc = f"Deputy {dep_id} | {leg_display} | Page {pagina} | Retrying {retry_count}/{max_retries} in {total_delay:.1f}s"
                            pbar.set_description(retry_desc)
                            pbar.write(f"🔄 Unexpected error retry {retry_count}/{max_retries} for ID {dep_id}, Leg {leg}, Page {pagina} after {total_delay:.1f}s")
                            time.sleep(total_delay)

                    # Stop paging if last request returned no results or failed out
                    if not success or (success and not page_data):
                        break
            
            # Update progress bar after completing all legislatures for this deputy
            pbar.update(1)
            pbar.write(f"✅ Deputy {dep_id} completed. Records collected: {len(all_data)}")

    # Final summary
    if all_data:
        df = pd.json_normalize(all_data)
        print(f"\n🎉 Scraping completed!")
        print(f"📊 Total records collected: {len(all_data)}")
        if failed_ids:
            print(f"❌ Failed requests: {len(failed_ids)}")
            for dep_id, leg, page in failed_ids:
                print(f"   - Deputy {dep_id}, Legislature {leg}, Page {page}")
        return df, failed_ids
    else:
        print("❌ No data fetched.")
        return None, failed_ids


# Example usage
df_discursos, failed_ids = fetch_deputados_discursos(
    [74171],
    id_legislatura=[52,53,54,55,56],
    itens=50,
    max_retries=10
)

if df_discursos is not None:
    df_discursos.to_csv('../data/raw/deputados_discursos.csv', index=False)

Deputy 74171 | Leg 55 | Page 2 | Retrying 1/10 in 1.1s:   0%|          | 0/1 [02:21<?, ?deputy/s]             

🔄 Retry 1/10 for ID 74171, Leg 55, Page 2 after 1.1s. Error: HTTPError


Deputy 74171 | Leg 55 | Page 2 | Retrying 2/10 in 2.1s:   0%|          | 0/1 [02:23<?, ?deputy/s]

🔄 Retry 2/10 for ID 74171, Leg 55, Page 2 after 2.1s. Error: HTTPError


Deputy 74171 | Leg 55 | Page 2 | Retrying 3/10 in 4.1s:   0%|          | 0/1 [02:26<?, ?deputy/s]

...

🔄 Retry 10/10 for ID 74171, Leg 55, Page 2 after 61.0s. Error: HTTPError


Deputy 74171 | Leg 56 (Leg 5/5) | Page 1 | Total Records: 3185:   0%|          | 0/1 [08:00<?, ?deputy/s]

❌ Failed after 10 retries for ID 74171, Leg 55, Page 2. Error: 500 Server Error: Internal Server Error for url: https://dadosabertos.camara.leg.br/api/v2/deputados/74171/discursos?ordenarPor=dataHoraInicio&ordem=DESC&itens=50&pagina=2&idLegislatura=55


Deputy 74171 | Leg 56 (Leg 5/5) | Page 1 | Found: 0 | Total: 3185: 100%|██████████| 1/1 [08:02<00:00, 482.07s/deputy]


✅ Deputy 74171 completed. Records collected: 3185

🎉 Scraping completed!
📊 Total records collected: 3185
❌ Failed requests: 1
   - Deputy 74171, Legislature 55, Page 2
